# Nazrul-4B Heretic — Camber L4 Iterative (`V1 → V2 → V3 → V4 → Final`) Notebook

Click **Run All** (or run cell-by-cell) on a Camber **`GPU • Xsmall` (NVIDIA L4 24GB)** node or any Linux/Cloud L4 GPU.
- **Base model:** `coder3101/gemma-4-E4B-it-heretic` (Heretic-decensored Gemma 4 E4B)
- **Dataset:** `2,534` unique Kazi Nazrul works (`3x`), `18` Bengali poetics/meter craft articles, `4,500` Bangla Wikipedia docs, `1,800` classic Bangla literature docs, `3,500` non-Nazrul poems, and `5,000` Bangla Alpaca-Orca instructions.
- **Training:** Iterative `V1 → V2 → V3 → V4 → Final` checkpoints saved under `outputs/versions/` and final fp16 adapter in `outputs/nazrul-lora/`.

In [ ]:
# 1. Check L4 GPU, pull latest `unsensored` code, and install dependencies (incl. brotli>=1.2.0 fix)
import os, subprocess, sys

# Optional: set your rotated HF token for faster Hub downloads (model is ungated)
# os.environ['HF_TOKEN'] = 'your_hf_token_here'

subprocess.run(['nvidia-smi'], check=False)
if not os.path.exists('src/train.py'):
    if not os.path.exists('NazrulLora'):
        subprocess.run(['git', 'clone', '-b', 'unsensored', 'https://github.com/ArafatAhmedMubinOfficial/NazrulLora.git'], check=True)
    os.chdir('NazrulLora')
else:
    subprocess.run(['git', 'checkout', 'unsensored'], check=False)
    subprocess.run(['git', 'pull', 'origin', 'unsensored'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'brotli>=1.2.0', 'brotlicffi>=1.2.0', 'urllib3>=2.2.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Workspace ready at:', os.getcwd())

In [ ]:
# 2. Build big Nazrul + Bengali Poetics + Modern Bangla dataset (~4-6 min)
import os, subprocess, sys
os.makedirs('data/cache', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

subprocess.run([
    sys.executable, '-u', 'src/prepare_data.py',
    '--cache-dir', 'data/cache',
    '--out-dir', 'data/processed'
], check=True)

In [ ]:
# 3. Train V1 -> V2 -> V3 -> V4 -> Final (~114 min total on L4 24GB)
# Streams live progress directly below; resumes from outputs/checkpoints/latest if needed.
import subprocess, sys

VERSIONS = 4  # Set 1, 2, 3, or 4
subprocess.run([
    sys.executable, '-u', 'src/train.py',
    '--preset', 'l4',
    '--data-dir', 'data/processed',
    '--out-dir', 'outputs',
    '--versions', str(VERSIONS),
    '--save-every', '50',
], check=True)

In [ ]:
# 4. Evaluate final adapter (base vs fine-tuned perplexity, rhyme rate, 8 sample prompts)
import json, subprocess, sys
from pathlib import Path

subprocess.run([
    sys.executable, '-u', 'src/eval_model.py',
    '--adapter-dir', 'outputs/nazrul-lora',
    '--val-jsonl', 'data/processed/cpt_val.jsonl',
    '--out-file', 'outputs/eval_report.json'
], check=True)

print('=== VERSION HISTORY ===')
vh = json.loads(Path('outputs/versions/version_history.json').read_text(encoding='utf-8'))
for item in vh:
    s2 = item.get('stage2') or {}
    print(f"{item['version'].upper()}: val_ppl={s2.get('val_ppl_after')} | elapsed={s2.get('elapsed_min')}m")

print('\n=== EVAL REPORT ===')
rep = json.loads(Path('outputs/eval_report.json').read_text(encoding='utf-8'))
print(f"Base PPL: {rep['heldout_nazrul_ppl_base']} -> Fine-tuned PPL: {rep['heldout_nazrul_ppl_finetuned']}")
for s in rep['samples']:
    print(f"\n--- {s['name']} (rhyme={s['rhyme_rate']}) ---\n{s['response']}")

In [ ]:
# 5. Test your own custom prompt interactively with best-of-4 Bengali rhyme selection
from pathlib import Path
from src.chat import load_nazrul_model, generate, score_bangla_rhyme

model, tok = load_nazrul_model(Path('outputs/nazrul-lora'))
prompt = 'কাজী নজরুল ইসলামের ভঙ্গিতে কঠোর ছন্দ ও অন্ত্যমিল বজায় রেখে আধুনিক সমাজের ভণ্ডামি নিয়ে ৮-১০ লাইনের একটি প্রতিবাদী কবিতা লেখো।'
reply = generate(model, tok, prompt, max_new_tokens=300, temperature=0.72, rhyme_candidates=4)
print(f'Rhyme score: {score_bangla_rhyme(reply):.2f}\n\n{reply}')